In [1]:
# تثبيت المكتبات اللازمة بأحدث الإصدارات
!pip install -q -U transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.1 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from google.colab import userdata
from huggingface_hub import login
import gc

# 1. Login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except:
    print("⚠️ Please ensure HF_TOKEN is set in Colab Secrets.")

# 2. Load Model
print("⏳ Loading MedGemma 4B... (This keeps the model in memory)")
processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it", token=hf_token)
model = AutoModelForImageTextToText.from_pretrained(
    "google/medgemma-1.5-4b-it",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)
print("✅ Model Ready!")

⏳ Loading MedGemma 4B... (This keeps the model in memory)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅ Model Ready!


In [3]:
# قاعدة بيانات المرضى
patients_db = [
    {
        "id": 1001, "age": 72, "gender": "M",
        "labs": "Creatinine 2.4 mg/dL (High - Stage 3 CKD)",
        "conditions": "Chronic Kidney Disease (CKD), Hypertension",
        "current_meds": "Lisinopril, Amlodipine",
        "proposed_drug": "Ibuprofen",
        "dose": "800 mg",
        "expected": "Danger (Kidney)"
    },
    {
        "id": 1002, "age": 65, "gender": "F",
        "labs": "Potassium 5.8 mEq/L (High)",
        "conditions": "Heart Failure",
        "current_meds": "Spironolactone, Lasix",
        "proposed_drug": "Lisinopril",
        "dose": "10 mg",
        "expected": "Danger (Hyperkalemia)"
    },
    {
        "id": 1003, "age": 55, "gender": "M",
        "labs": "INR Variable",
        "conditions": "Atrial Fibrillation",
        "current_meds": "Warfarin",
        "proposed_drug": "Aspirin",
        "dose": "325 mg",
        "expected": "Warning (Bleeding Risk)"
    },
    {
        "id": 1004, "age": 40, "gender": "F",
        "labs": "Renal Function: Normal",
        "conditions": "Asthma",
        "current_meds": "Albuterol Inhaler",
        "proposed_drug": "Amoxicillin",
        "dose": "500 mg",
        "expected": "Safe"
    },
    {
        "id": 1005, "age": 81, "gender": "M",
        "labs": "Creatinine 1.9 mg/dL (High)",
        "conditions": "Type 2 Diabetes",
        "current_meds": "Insulin Glargine",
        "proposed_drug": "Metformin",
        "dose": "1000 mg",
        "expected": "Danger (Lactic Acidosis)"
        # Metformin is contraindicated if Creatinine is high/eGFR<30, so Danger is actually correct medically
    }
]

# قاعدة المعرفة الدوائية (محسنة جداً)
drug_kb = {
    "Ibuprofen": """
    NSAID.
    - CONTRAINDICATED (DANGER) in: Chronic Kidney Disease (CKD), High Creatinine, or Heart Failure.
    - INTERACTION (WARNING) with: Anticoagulants (increases bleeding risk).
    """,
    "Lisinopril": """
    ACE Inhibitor.
    - CONTRAINDICATED (DANGER) if: Potassium > 5.0 mEq/L (Hyperkalemia risk) or history of Angioedema.
    - MONITOR (WARNING): When combined with Spironolactone.
    """,
    "Aspirin": """
    Antiplatelet.
    - INTERACTION (WARNING): Combined with Warfarin greatly increases bleeding risk (Monitor INR).
    - CONTRAINDICATED (DANGER): In active GI bleeding or nasal polyps with asthma.
    """,
    "Amoxicillin": """
    Antibiotic (Penicillin class).
    - CONTRAINDICATED (DANGER): Only if patient has history of PENICILLIN ANAPHYLAXIS.
    - SAFE: For most patients with Asthma (unless specific allergy noted).
    """,
    "Metformin": """
    Antidiabetic.
    - CONTRAINDICATED (DANGER): If Serum Creatinine is High (Males > 1.5, Females > 1.4) or eGFR < 30 (Risk of Fatal Lactic Acidosis).
    """
}

In [4]:
import re
from PIL import Image

def analyze_patient_case(patient):
    dummy_image = Image.new('RGB', (224, 224), color='black')
    kb_info = drug_kb.get(patient['proposed_drug'], "General precaution.")

    # هندسة البرومبت المتقدمة
    prompt_text = f"""
    Role: Expert Clinical Pharmacist System.

    === PATIENT CASE ===
    • Age: {patient['age']} | Gender: {patient['gender']}
    • Conditions: {patient['conditions']}
    • Labs: {patient['labs']}
    • Current Meds: {patient['current_meds']}

    === DRUG REQUEST ===
    • Name: {patient['proposed_drug']}
    • Dose: {patient['dose']}

    === PHARMACOLOGY GUIDELINES ===
    {kb_info}

    === DECISION RULES ===
    1. SAFE: No significant interactions or contraindications.
    2. WARNING: Drug interaction exists (e.g., bleeding risk) but usage might be allowed with monitoring.
    3. DANGER: Strict CONTRAINDICATION (e.g., Kidney failure + NSAIDs, High Potassium + ACE inhibitors, High Creatinine + Metformin).

    === INSTRUCTIONS ===
    Analyze the case. If a specific contraindication is found in the guidelines matching the patient's labs/conditions, you MUST choose DANGER.

    OUTPUT FORMAT:
    Output ONLY the result in brackets at the end.
    Example: [[ DANGER | Patient has CKD ]]

    YOUR ANALYSIS:
    """

    messages = [{"role": "user", "content": [{"type": "image", "image": dummy_image}, {"type": "text", "text": prompt_text}]}]

    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=500, do_sample=False) # 500 كافية جداً الآن
        generation = generation[0][input_len:]

    full_output = processor.decode(generation, skip_special_tokens=True)

    # استخراج النتيجة
    matches = re.findall(r"\[\[\s*(.*?)\s*\|\s*(.*?)\s*\]\]", full_output, re.DOTALL)

    if matches:
        # نأخذ آخر نتيجة صالحة لا تحتوي على كلمة Example أو Status
        for status, reason in reversed(matches):
            if "STATUS" not in status and "DANGER" not in reason: # فلترة إضافية
                return status.upper().strip(), reason.replace("\n", " ").strip()

    # في حال الفشل
    return "ERROR", full_output[-100:]

# --- التشغيل ---
print("\n" + "="*120)
print(f"{'Drug':<12} | {'Expected Label':<25} | {'AI Decision':<15} | {'AI Reasoning'}")
print("="*120)

for p in patients_db:
    torch.cuda.empty_cache()
    status, reason = analyze_patient_case(p)

    # أيقونات
    icon = "⚪"
    if "DANGER" in status: icon = "🔴"
    elif "WARNING" in status: icon = "🟡"
    elif "SAFE" in status: icon = "🟢"

    # تنسيق السبب
    reason_short = (reason[:75] + '..') if len(reason) > 75 else reason

    print(f"{p['proposed_drug']:<12} | {p['expected']:<25} | {icon} {status:<12} | {reason_short}")
    print("-" * 120)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.



Drug         | Expected Label            | AI Decision     | AI Reasoning


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Ibuprofen    | Danger (Kidney)           | 🔴 DANGER       | Patient has CKD
------------------------------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Lisinopril   | Danger (Hyperkalemia)     | 🔴 DANGER       | Patient has high potassium (5.8 mEq/L) and is taking Spironolactone, an ACE..
------------------------------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Aspirin      | Warning (Bleeding Risk)   | 🟡 WARNING      | Aspirin interaction with Warfarin increases bleeding risk (Monitor INR)
------------------------------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Amoxicillin  | Safe                      | 🟢 SAFE         | No contraindications identified
------------------------------------------------------------------------------------------------------------------------
Metformin    | Danger (Lactic Acidosis)  | 🔴 DANGER       | Patient has CKD (Creatinine 1.9 mg/dL)
------------------------------------------------------------------------------------------------------------------------


In [ ]:
import json
from google.colab import drive

# 1. احفظ النوت بوك الحالي لضمان وجود أحدث نسخة
# (اضغط Ctrl+S يدوياً قبل تشغيل هذه الخلية)

# 2. قراءة ملف النوت بوك الحالي
# ملاحظة: في كولاب، الملف الحالي لا يمكن الوصول إليه مباشرة بسهولة كملف
# لذا سنقوم بحيلة بسيطة: تنزيل المحتوى الحالي كملف JSON وتنظيفه
from google.colab import _message
nb_json = _message.blocking_request('get_ipynb')

# 3. حذف بيانات الـ widgets المسببة للمشكلة
if 'widgets' in nb_json['ipynb']['metadata']:
    del nb_json['ipynb']['metadata']['widgets']
    print("✅ تم حذف بيانات widgets بنجاح.")
else:
    print("ℹ️ لم يتم العثور على widgets (الملف نظيف أصلاً).")

# 4. حفظ الملف النظيف باسم جديد
clean_filename = "MedGemma_Cleaned.ipynb"
with open(clean_filename, 'w', encoding='utf-8') as f:
    json.dump(nb_json['ipynb'], f, indent=1)

# 5. تحميل الملف النظيف تلقائياً لجهازك
from google.colab import files
files.download(clean_filename)